# 00 - SBM Validation Map

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [SBM package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: canonical SBM sensitivity rows, or one Arrow table per homogeneous `(risk_class, risk_measure)` path, with deterministic sensitivity identifiers, desk and legal-entity lineage, bucket labels, risk-factor vertices, and signed sensitivity amounts. The machine-readable GIRR delta input-table example is [`frtb_sbm.girr_delta.schema.json`](../../../docs/schemas/input_table/frtb_sbm.girr_delta.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

This notebook maps the current `frtb-sbm` public calculation surface. It uses the Basel MAR21 profile implemented by the package and the executable fixture packs kept with the package tests. The examples are synthetic development artifacts and are not final regulatory capital outputs.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None
for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_sbm").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    if (candidate / "packages" / "frtb-sbm" / "src" / "frtb_sbm").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = candidate / "packages" / "frtb-sbm"
        break
if PACKAGE_ROOT is None:
    raise RuntimeError("Could not locate frtb-sbm package root")
workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src"))) if REPO_ROOT else ()
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    if path is not None and str(path) not in sys.path:
        sys.path.insert(0, str(path))
PACKAGE_ROOT

In [ ]:
from collections import Counter

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


from frtb_sbm import SbmRegulatoryProfile, get_sbm_rule_profile, supported_risk_class_measures
from sbm_notebook_data import FIXTURE_PACKS, markdown_table

## Public API happy path

This notebook starts by using top-level SBM public APIs to enumerate the supported Basel MAR21 risk-class and risk-measure paths.


## SBM validation workflow

```mermaid
flowchart LR
    A[Committed SBM fixtures] --> B[Public fixture loaders]
    B --> C[Rule profile and support matrix]
    B --> D[Delta fixture notebooks]
    B --> E[Vega and curvature notebooks]
    B --> F[Arrow handoff notebook]
    C --> G[Notebook validation map]
    D --> G
    E --> G
    F --> G
```


In [ ]:
profile = get_sbm_rule_profile(SbmRegulatoryProfile.BASEL_MAR21)
paths = sorted(
    supported_risk_class_measures(profile.profile_id),
    key=lambda item: (item[0].value, item[1].value),
)

display(
    Markdown(
        markdown_table(
            ("Profile", "Regulator", "Version", "Content hash"),
            (
                (
                    profile.profile_id,
                    profile.regulator,
                    profile.version,
                    profile.content_hash[:16] + "...",
                ),
            ),
        )
    )
)

## Implementation details / math verification - Fixture and notebook map

The remaining cells map the executable fixture packs and notebook coverage back to the supported paths.


In [ ]:
display(
    Markdown(
        markdown_table(
            ("Risk class", "Risk measure"),
            ((risk_class.value, measure.value) for risk_class, measure in paths),
        )
    )
)

In [ ]:
measure_counts = Counter(measure.value for _risk_class, measure in paths)
display(
    Markdown(
        markdown_table(
            ("Measure", "Supported risk classes"),
            ((measure, measure_counts[measure]) for measure in sorted(measure_counts)),
        )
    )
)

In [ ]:
display(
    Markdown(
        markdown_table(
            ("Fixture", "Risk class", "Risk measure", "What it demonstrates"),
            (
                (fixture.fixture_id, fixture.risk_class, fixture.risk_measure, fixture.description)
                for fixture in FIXTURE_PACKS
            ),
        )
    )
)

In [ ]:
display(
    Markdown(
        markdown_table(
            ("Notebook", "Purpose"),
            (
                ("00_validation_map", "Supported profile, capital paths, and fixture inventory."),
                (
                    "01_delta_fixture_walkthrough",
                    "GIRR delta replay, audit hashes, weighted sensitivities, and invalid-case failure.",
                ),
                (
                    "02_vega_curvature_paths",
                    "GIRR/non-GIRR vega fixture replay plus curvature branch evidence.",
                ),
                ("03_arrow_batch_fast_path", "Arrow batch usage and fast-path diagnostics."),
            ),
        )
    )
)

## See also

- [SBM package journey](../docs/PACKAGE_JOURNEY.md)
- [SBM dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
